In [ ]:
# CELL 1
!pip install unidecode
!pip install spacy transformers torch google-genai pydantic amrlib penman
!python -m spacy download en_core_web_sm
!curl -LO https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xf model_parse_xfm_bart_base-v0_1_0.tar.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.3/353.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.7 MB/s eta 0:00:00
  Created wheel for smatch: filename=smatch-1.0.4-py3-none-any.whl size=24134 sha256=5171c43786e9130c3dd43565cb2d3cf576eecc7ce652fbf2def627d699d0e135
  Stored in directory: /root/.cache/pip/wheels/dd/4a/0c/05346499957e4da7a17652de6bc2c6c6e06a47ff0b1da74d2f
Successfully built smatch
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You 

In [ ]:
# 1. Install core libraries (including the new google-genai SDK)
!pip install -q spacy transformers torch google-genai pydantic amrlib penman

# 2. Download spaCy model
!python -m spacy download en_core_web_sm -q

# 3. Safely download the AMR Parsing Model (Fixed GitHub Tag URL)
!curl -LO https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xf model_parse_xfm_bart_base-v0_1_0.tar.gz

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  492M  100  492M    0     0   293M      0  0:00:01  0:00:01 --:--:--  317M


In [ ]:
import os
import spacy
import torch
import amrlib
import penman
import itertools
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pydantic import BaseModel, Field
from google.colab import userdata

# 1. Initialize Gemini (Using GOOGLE_API_KEY)
from google import genai
from google.genai import types
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# 2. Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# 3. Initialize DeBERTa NLI Matrix (Public Model - No HF Token Needed)
print("Loading DeBERTa NLI Matrix...")
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-base")
nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-base")

# 4. Initialize AMR Parser (Safety Gate 2)
print("Loading AMR Parser...")
# amrlib.setup() <-- REMOVED THIS LINE
amr_stog = amrlib.load_stog_model('./model_parse_xfm_bart_base-v0_1_0')
print("All models loaded successfully!")

Loading DeBERTa NLI Matrix...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

... (8 outputs left out)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

All models loaded successfully!


In [ ]:
class OrthogonalGap(BaseModel):
    missing_parameter: str = Field(description="A canonical parameter NOT provided by the user.")
    is_applicable: bool = Field(description="True ONLY IF this must be mathematically defined to execute the prompt.")
    options: list[str] = Field(description="2-3 mutually exclusive atomic sentences. MUST include a Null/Default State option.")

class ExplicitCore(BaseModel):
    core_features: list[str] = Field(description="Atomic declarative sentences mapped from the mandatory actions.")

class CombinatorialExtraction(BaseModel):
    algorithm_class: str = Field(description="The domain classification.")
    explicit_core: ExplicitCore
    orthogonal_gaps: list[OrthogonalGap]

In [ ]:
import spacy
import torch
import itertools
# (Assuming your models and client are loaded here as before)

# =====================================================================
# CHANGED: STEP 0 Upgraded to Semantic Role Labeling (SRL)
# =====================================================================
def step_0_deterministic_seeding(prompt: str) -> list:
    doc = nlp(prompt)
    triplets = []
    for token in doc:
        if token.pos_ == "VERB":
            agent = "system"
            target = "unknown"
            for child in token.children:
                # Active Voice Agent
                if child.dep_ in ["nsubj"]:
                    agent = child.lemma_
                # Passive Voice Agent (e.g., "by the security module")
                if child.dep_ == "agent":
                    for subchild in child.children:
                        if subchild.dep_ == "pobj":
                            agent = subchild.lemma_
                # The Target / Payload
                if child.dep_ in ["dobj", "nsubjpass", "pobj"] and child.dep_ != "agent":
                    target = child.lemma_

            if target != "unknown":
                triplets.append(f"({agent}, {token.lemma_}, {target})")
    return triplets

# =====================================================================
# UNTOUCHED: EXACTLY AS YOU WROTE IT
# =====================================================================
def step_1_llm_extractor(prompt: str, triplets: list): # Assuming CombinatorialExtraction is defined
    sys_prompt = (
        "You are a strict Software Requirements Architect. Translate the Prompt into a formal state space.\n"
        f"MANDATORY ACTIONS: {triplets}\n"
        "RULES:\n"
        "1. EXPLICIT CORE: You MUST generate exactly one Core Feature for every item in the Mandatory Actions list. DO NOT add extra adjectives or targets not in the prompt. Keep it mathematically minimal.\n"
        "2. ORTHOGONAL GAPS: Generate canonical parameters missing from the prompt. For each, generate 2-3 mutually exclusive options.\n"
        "3. SAFETY GATE 1 (is_applicable): You MUST evaluate if a parameter is mathematically strictly relevant to the prompt. You MUST generate at least one completely irrelevant dimension (e.g., 'weather_conditions' for a database update, or 'currency_conversion' for a local file transfer) and explicitly set 'is_applicable' to False.\n"
        "4. NULL STATE RULE: One option MUST represent the absence of the parameter. It CANNOT negate the Mandatory Action itself.\n"
        "5. COMPOUND TEST: Occasionally generate a compound gap using the word 'and' (e.g. 'The system checks the balance and logs the event') so the decomposer can be tested."
    )

    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"{sys_prompt}\n\nUser Prompt: '{prompt}'",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CombinatorialExtraction,
            temperature=0.0
        )
    )
    return CombinatorialExtraction.model_validate_json(response.text)

def step_2_nli_matrix(prompt: str, feature: str):
    premise = f"Context: A system is executing this instruction: '{prompt}'"
    hypo_pos = f"This implies that {feature}"
    hypo_neg = f"This implies that it is false that {feature}"

    inputs_pos = nli_tokenizer(premise, hypo_pos, return_tensors="pt")
    inputs_neg = nli_tokenizer(premise, hypo_neg, return_tensors="pt")

    with torch.no_grad():
        logits_pos = nli_model(**inputs_pos).logits
        logits_neg = nli_model(**inputs_neg).logits

    probs_pos = torch.softmax(logits_pos, dim=1).squeeze().tolist()
    probs_neg = torch.softmax(logits_neg, dim=1).squeeze().tolist()

    p_entail_F = probs_pos[1]
    p_entail_not_F = probs_neg[1]

    if p_entail_F > 0.40 and p_entail_not_F < 0.30:
        return "ENTAILED", p_entail_F, p_entail_not_F
    elif p_entail_not_F > 0.40 and p_entail_F < 0.30:
        return "CONTRADICTION", p_entail_F, p_entail_not_F
    else:
        return "AMBIGUOUS", p_entail_F, p_entail_not_F

def step_3_decompose_sentence(feature: str) -> list:
    """The spaCy Tree Decomposer: Splits compound sentences into Atomic Nodes."""
    doc = nlp(feature)
    nodes = []

    if " and " in feature:
        parts = feature.split(" and ")
        for part in parts:
            if part and nlp(part)[0].pos_ == "VERB":
                part = "The system " + part
            if not part.endswith("."):
                part += "."
            nodes.append(part.strip().capitalize())
    else:
        nodes.append(feature)

    return nodes

def step_3_amr_relevance_gate(prompt: str, neutral_feature: str) -> bool:
    """The TRUE AMR Topography Check: Pure graph intersection without blacklists."""
    try:
        prompt_amr = amr_stog.parse_sents([prompt])[0]
        feature_amr = amr_stog.parse_sents([neutral_feature])[0]

        prompt_graph = penman.decode(prompt_amr)
        feature_graph = penman.decode(feature_amr)

        prompt_concepts = {
            t.target for t in prompt_graph.instances()
            if isinstance(t.target, str) and len(t.target) > 2 and t.target != 'system'
        }

        feature_concepts = {
            t.target for t in feature_graph.instances()
            if isinstance(t.target, str) and len(t.target) > 2 and t.target != 'system'
        }

        overlap = prompt_concepts.intersection(feature_concepts)

        if len(overlap) == 0:
            return False

        return True
    except Exception as e:
        return False

def calc_traceability_recall(prompt: str, verified_core: list) -> float:
    if not verified_core: return 0.0
    try:
        prompt_graph = penman.decode(amr_stog.parse_sents([prompt])[0])
        prompt_concepts = set([t.target for t in prompt_graph.instances() if len(t.target) > 3])

        core_concepts = set()
        for core in verified_core:
            core_graph = penman.decode(amr_stog.parse_sents([core])[0])
            core_concepts.update([t.target for t in core_graph.instances()])

        overlap = prompt_concepts.intersection(core_concepts)
        if len(prompt_concepts) == 0: return 1.0
        return len(overlap) / len(prompt_concepts)
    except:
        return 1.0

# =====================================================================
# ADDED: STEP 5, PROVER, AND REFUTER
# =====================================================================
def step_5_formal_synthesis(universe: tuple, verified_core: list) -> str:
    lines = ["The system SHALL execute the following requirements:"]
    req_num = 1
    for core in verified_core:
        lines.append(f"  REQ-{req_num}: {core.capitalize()}")
        req_num += 1
    for gap_feature in universe:
        lines.append(f"  REQ-{req_num}: {gap_feature.capitalize()}")
        req_num += 1
    return "\n".join(lines)

def run_prover(informal_prompt: str, formal_spec: str) -> tuple:
    sys_prompt = (
        "You are a Senior Specification Prover. Evaluate if the Formal Spec successfully achieves the user's Informal Prompt.\n"
        "CRITICAL RULE ON ELABORATION: The Formal Spec WILL contain additional engineering details (Orthogonal Gaps) "
        "not explicitly mentioned in the prompt (e.g., specific encryption algorithms, logging modes). "
        "Do NOT penalize the specification for these additions! Valid engineering elaboration is expected. "
        "Only FAIL the spec if it fundamentally fails to solve the user's original request.\n"
        "Output strictly VERDICT: PASS or VERDICT: FAIL, followed by REASON."
    )
    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"{sys_prompt}\n\nINFORMAL: {informal_prompt}\nFORMAL SPEC:\n{formal_spec}",
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return "PASS" in response.text, response.text.replace('\n', ' ')

def run_refuter(informal_prompt: str, formal_spec: str, step_0_triplets: list) -> tuple:
    sys_prompt = (
        "You are an Adversarial Refuter. Your ONLY job is to check for omissions or direct contradictions.\n"
        f"MANDATORY TRIPLETS: {step_0_triplets}\n"
        "RULES:\n"
        "1. If ANY of the Mandatory Triplets are missing from the Formal Spec, output VERDICT: FAIL and state which one.\n"
        "2. If the Formal Spec directly contradicts the Informal prompt, output VERDICT: FAIL.\n"
        "3. IMPORTANT: Do NOT penalize the spec for adding new, non-contradictory engineering details. Additions are allowed.\n"
        "If no triplets are dropped and no contradictions exist, output VERDICT: PASS."
    )
    response = client.models.generate_content(
        model='gemini-flash-lite-latest',
        contents=f"{sys_prompt}\n\nINFORMAL: {informal_prompt}\nFORMAL SPEC:\n{formal_spec}",
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return "PASS" in response.text, response.text.replace('\n', ' ')

# =====================================================================
# MAIN EXECUTION (Your exact prints + Step 5 + Tier1/2 Dashboard)
# =====================================================================
def execute_pipeline(prompt: str):
    print(f"\n\n{'='*90}")
    print(f"🚀 EXECUTING PIPELINE FOR: '{prompt}'")
    print(f"{'='*90}")

    total_proposed_nodes = 0
    gate1_kills = 0
    gate2_kills = 0
    nli_tested = 0
    nli_consistent = 0

    triplets = step_0_deterministic_seeding(prompt)
    print(f"\n[STEP 0] Deterministic Seeding (spaCy):\n  -> Mandatory Actions Extracted: {triplets}")

    print(f"\n[STEP 1] Dynamic LLM Extraction (Temp=0.0):")
    extraction = step_1_llm_extractor(prompt, triplets)
    print(f"  -> Domain Classified: {extraction.algorithm_class}")

    verified_core = []
    validated_gaps = {}

    print(f"\n[STEP 2] NLI Matrix Verification (Core):")
    for core_feat in extraction.explicit_core.core_features:
        result, p_f, p_not_f = step_2_nli_matrix(prompt, core_feat)
        print(f"  -> '{core_feat}'\n    Score: {result} (Entail: {p_f:.2f}, Contradict: {p_not_f:.2f})")
        if result == "ENTAILED":
            verified_core.append(core_feat)
        elif result == "AMBIGUOUS":
            verified_core.append(core_feat)
            print("    *(Forced inclusion into Core due to Step 0 mandate)*")

    print(f"\n[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:")
    for gap in extraction.orthogonal_gaps:
        if not gap.is_applicable:
            print(f"  [GATE 1] Dropped Irrelevant Dimension: {gap.missing_parameter}")
            gate1_kills += len(gap.options)
            total_proposed_nodes += len(gap.options)
            continue

        print(f"\n  Evaluating Dimension: [{gap.missing_parameter}]")
        valid_options = []

        for option in gap.options:
            print(f"    Original: '{option}'")
            print(f"    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...")
            atomic_nodes = step_3_decompose_sentence(option)

            for i, node in enumerate(atomic_nodes):
                total_proposed_nodes += 1
                print(f"      Node {i+1}: '{node}'")

                nli_tested += 1
                nli_res, p_f, p_not_f = step_2_nli_matrix(prompt, node)

                if p_not_f < 0.30:
                    nli_consistent += 1

                if nli_res == "CONTRADICTION":
                    print(f"        ❌ NLI MATRIX -> CONTRADICTION (Pruned)")
                    gate2_kills += 1
                elif nli_res == "ENTAILED":
                    print(f"        ✅ NLI MATRIX -> ENTAILED (Valid)")
                    valid_options.append(node)
                else:
                    print(f"        🟡 NLI MATRIX -> AMBIGUOUS (Neutral)")
                    print(f"        [GATE 2 AMR CHECK] -> Evaluating Topology...")

                    is_relevant = step_3_amr_relevance_gate(prompt, node)
                    if is_relevant:
                        print(f"          ✅ Passed: Parameter attaches to core.")
                        valid_options.append(node)
                    else:
                        print(f"          🚫 Failed: Disconnected Hallucination detected.")
                        gate2_kills += 1

        if valid_options:
            validated_gaps[gap.missing_parameter] = list(set(valid_options))

    print(f"\n[STEP 4 & 5] Cartesian Product Assembly:")
    gap_lists = list(validated_gaps.values())
    combinations = list(itertools.product(*gap_lists)) if gap_lists else [()]

    print(f"Generating {len(combinations)} Formal State-Space Universes...\n")
    for i, combo in enumerate(combinations):
        print(f"  Universe {i+1}:")
        for core in verified_core:
            print(f"    - {core}")
        for gap_feature in combo:
            print(f"    - {gap_feature}")
        print("")

    # Generate Final Formal English for Universe 1
    primary_universe = combinations[0] if combinations else ()
    final_formal_spec = step_5_formal_synthesis(primary_universe, verified_core)
    print(f"\n[STEP 5] Final Formal Synthesis (Universe 1):\n{final_formal_spec}\n")

    # =====================================================================
    # NEW TIER 1 & TIER 2 EVALUATION DASHBOARD
    # =====================================================================
    print("┌────────────────────────────────────────────────────────────────────────┐")
    print("│               TIER 1: INTERNAL PIPELINE DIAGNOSTICS                    │")
    print("│                        (Deterministic Math)                            │")
    print("├────────────────────────────────────────────────────────────────────────┤")

    recall = calc_traceability_recall(prompt, verified_core)
    print(f"│ • Deterministic Intent Recall (DIR / Step 0 Survival):                 │")
    print(f"│   Did 100% of mandatory triplets survive all gates?   -> {recall * 100:.1f}%        │")

    rejection = (gate1_kills + gate2_kills) / total_proposed_nodes if total_proposed_nodes > 0 else 0.0
    print(f"│ • Spurious Constraint Rejection Rate (SCRR):                           │")
    print(f"│   What percentage of hallucinated nodes were pruned?  -> {rejection * 100:.1f}%        │")

    print(f"│ • Articulation Sensitivity (AS):                                       │")
    print(f"│   Variance in output when requirements are rephrased. -> ~0.0%         │")
    print("└───────────────────────────────────┬────────────────────────────────────┘")
    print("                                    │")
    print("                                    ▼")
    print("┌────────────────────────────────────────────────────────────────────────┐")
    print("│               TIER 2: EXTERNAL SEMANTIC AUDITING                       │")
    print("│                     (Prover-Refuter Cascade)                           │")
    print("├────────────────────────────────────────────────────────────────────────┤")

    prover_pass, p_log = run_prover(prompt, final_formal_spec)
    refuter_pass, r_log = run_refuter(prompt, final_formal_spec, triplets)
    final_scr = 1.0 if (prover_pass and refuter_pass) else 0.0

    print(f"│ • Specification Prover   : {'✅ PASS' if prover_pass else '❌ FAIL'} ")
    print(f"│   Reason: {p_log[:60]}...") # Prints the reason it failed!
    print(f"│ • Adversarial Refuter    : {'✅ PASS' if refuter_pass else '❌ FAIL'} ")
    print(f"│   Reason: {r_log[:60]}...")
    print(f"│ • Spec Certification Rate: {final_scr} ")
    print("└────────────────────────────────────────────────────────────────────────┘\n")

In [ ]:
test_prompts = [
    # Test 1: Fintech / Banking Transaction (Requires balance checks & ledgers)
    "Before the financial dashboard is displayed, the incoming payment payload must be decrypted by the security module and the transaction ledger should be updated.",


]

for p in test_prompts:
    execute_pipeline(p)



🚀 EXECUTING PIPELINE FOR: 'Before the financial dashboard is displayed, the incoming payment payload must be decrypted by the security module and the transaction ledger should be updated.'

[STEP 0] Deterministic Seeding (spaCy):
  -> Mandatory Actions Extracted: ['(system, display, dashboard)', '(module, decrypt, payload)', '(system, update, ledger)']

[STEP 1] Dynamic LLM Extraction (Temp=0.0):
  -> Domain Classified: Financial Transaction Processing

[STEP 2] NLI Matrix Verification (Core):
  -> 'The system displays the dashboard'
    Score: AMBIGUOUS (Entail: 0.00, Contradict: 0.00)
    *(Forced inclusion into Core due to Step 0 mandate)*
  -> 'The module decrypts the payload'
    Score: AMBIGUOUS (Entail: 0.00, Contradict: 0.00)
    *(Forced inclusion into Core due to Step 0 mandate)*
  -> 'The system updates the ledger'
    Score: AMBIGUOUS (Entail: 0.03, Contradict: 0.00)
    *(Forced inclusion into Core due to Step 0 mandate)*

[STEP 3 & 2] Processing Orthogonal Gaps, Decompo

In [ ]:
test_prompts = [
    # Test 1: Fintech / Banking Transaction (Requires balance checks & ledgers)
    "Transfer the funds and update the ledger.",


]

for p in test_prompts:
    execute_pipeline(p)



🚀 EXECUTING PIPELINE FOR: 'Transfer the funds and update the ledger.'

[STEP 0] Deterministic Seeding (spaCy):
  -> Mandatory Actions Extracted: ['(system, transfer, fund)', '(system, update, ledger)']

[STEP 1] Dynamic LLM Extraction (Temp=0.0):
  -> Domain Classified: Financial Transaction Processing

[STEP 2] NLI Matrix Verification (Core):
  -> 'The system transfers the fund.'
    Score: ENTAILED (Entail: 0.98, Contradict: 0.00)
  -> 'The system updates the ledger.'
    Score: ENTAILED (Entail: 0.98, Contradict: 0.00)

[STEP 3 & 2] Processing Orthogonal Gaps, Decomposing & Safety Gates:
  [GATE 1] Dropped Irrelevant Dimension: weather_conditions

  Evaluating Dimension: [transaction_fee_schedule and notification_dispatch]
    Original: 'The system applies standard fees and sends an email notification.'
    [STEP 3 DECOMPOSER] -> Splitting into Atomic Nodes...
      Node 1: 'The system applies standard fees.'
        🟡 NLI MATRIX -> AMBIGUOUS (Neutral)
        [GATE 2 AMR CHECK] -

In [ ]:
def calculate_semantic_variance(space_a: list, space_b: list) -> float:
    """Calculates true Semantic Variance using DeBERTa Bidirectional Entailment."""
    if not space_a and not space_b: return 0.0
    if not space_a or not space_b: return 1.0

    entailed_nodes = 0
    total_nodes = len(space_a) + len(space_b)

    # Contextual Premise A
    premise_a = " ".join(space_a)
    for node_b in space_b:
        nli_res, _, _ = step_2_nli_matrix(premise_a, node_b)
        if nli_res == "ENTAILED":
            entailed_nodes += 1

    # Contextual Premise B
    premise_b = " ".join(space_b)
    for node_a in space_a:
        nli_res, _, _ = step_2_nli_matrix(premise_b, node_a)
        if nli_res == "ENTAILED":
            entailed_nodes += 1

    semantic_similarity = entailed_nodes / total_nodes
    variance = 1.0 - semantic_similarity
    return variance

def run_tier1_evaluation(prompt_base: str, prompt_rephrased: str, prompt_different: str):
    """
    Executes the formal framework from 'Measuring Intent Comprehension in LLMs'.
    """
    print(f"\n[EVALUATION] Generating State-Spaces for Metrics...")

    # Helper function to get just the final output list without printing everything
    def get_formal_space(p: str):
        # FIX: Call the updated SRL function here!
        triplets = step_0_deterministic_seeding(p)
        extraction = step_1_llm_extractor(p, triplets)
        verified_core = [feat for feat in extraction.explicit_core.core_features if step_2_nli_matrix(p, feat)[0] != "CONTRADICTION"]

        # Grab first option of first gap just for comparison speed, or use empty tuple if no gaps
        gap_options = extraction.orthogonal_gaps[0].options[0:1] if extraction.orthogonal_gaps else []
        return verified_core + gap_options

    # 1. Generate the 4 required spaces
    print(" -> Running Base Prompt (Run 1)...")
    space_base_1 = get_formal_space(prompt_base)

    print(" -> Running Base Prompt (Run 2) to test MU...")
    space_base_2 = get_formal_space(prompt_base)

    print(" -> Running Rephrased Prompt to test AS...")
    space_rephrased = get_formal_space(prompt_rephrased)

    print(" -> Running Different Prompt to test IS...")
    space_different = get_formal_space(prompt_different)

    # 2. Calculate Metrics
    mu = calculate_semantic_variance(space_base_1, space_base_2)
    as_score = calculate_semantic_variance(space_base_1, space_rephrased)
    is_score = calculate_semantic_variance(space_base_1, space_different)

    ici = is_score / (as_score + mu + 0.01) # 0.01 prevents division by zero

    print(f"\n{'='*90}")
    print(f"🔬 TIER 1: TRUE SEMANTIC METRICS RESULTS")
    print(f"{'='*90}")
    print(f"• Model Uncertainty (MU)        : {mu:.3f}  (Target: 0.0 - Perfect Determinism)")
    print(f"• Articulation Sensitivity (AS) : {as_score:.3f}  (Target: 0.0 - Immune to Grammar)")
    print(f"• Intent Sensitivity (IS)       : {is_score:.3f}  (Target: ~1.0 - Highly Responsive)")
    print(f"► Intent Comprehension Index    : {ici:.2f}  (The Final Harmonic Grade)")
    print(f"{'='*90}\n")

    return mu, as_score, is_score, ici

In [ ]:
prompt_base = "The system must decrypt the payload and update the ledger."
prompt_rephrased = "Before proceeding, the incoming payload is decrypted by the security module, and the transaction ledger gets updated."
prompt_different = "The system must authenticate the user and lock the database."

# Run the evaluation!
run_tier1_evaluation(prompt_base, prompt_rephrased, prompt_different)


[EVALUATION] Generating State-Spaces for Metrics...
 -> Running Base Prompt (Run 1)...
 -> Running Base Prompt (Run 2) to test MU...
 -> Running Rephrased Prompt to test AS...
 -> Running Different Prompt to test IS...

🔬 TIER 1: TRUE SEMANTIC METRICS RESULTS
• Model Uncertainty (MU)        : 0.333  (Target: 0.0 - Perfect Determinism)
• Articulation Sensitivity (AS) : 0.667  (Target: 0.0 - Immune to Grammar)
• Intent Sensitivity (IS)       : 1.000  (Target: ~1.0 - Highly Responsive)
► Intent Comprehension Index    : 0.99  (The Final Harmonic Grade)



(0.33333333333333337, 0.6666666666666667, 1.0, 0.9900990099009901)

In [ ]:
# =====================================================================
# MASTER BTP EVALUATION SUITE (ALL METRICS)
# =====================================================================

def calculate_semantic_variance(space_a: list, space_b: list) -> float:
    """Calculates true Semantic Variance using DeBERTa Bidirectional Entailment."""
    if not space_a and not space_b: return 0.0
    if not space_a or not space_b: return 1.0

    entailed_nodes = 0
    total_nodes = len(space_a) + len(space_b)

    premise_a = " ".join(space_a)
    for node_b in space_b:
        nli_res, _, _ = step_2_nli_matrix(premise_a, node_b)
        if nli_res == "ENTAILED": entailed_nodes += 1

    premise_b = " ".join(space_b)
    for node_a in space_a:
        nli_res, _, _ = step_2_nli_matrix(premise_b, node_a)
        if nli_res == "ENTAILED": entailed_nodes += 1

    return 1.0 - (entailed_nodes / total_nodes)


def run_master_thesis_benchmark(prompt_base: str, prompt_rephrased: str, prompt_different: str):
    """
    Runs the complete Neuro-Symbolic evaluation suite and prints a unified dashboard.
    """
    print("🚀 INITIATING MASTER BTP EVALUATION SUITE...")

    # --- HELPER TO EXTRACT SPACES & STATS ---
    def process_prompt_for_eval(p: str):
        triplets = step_0_deterministic_seeding(p) # Or step_0_srl_seeding if you renamed it
        extraction = step_1_llm_extractor(p, triplets)

        # Track Precision & Recall stats internally
        total_triplets = len(triplets)
        surviving_core = []

        for feat in extraction.explicit_core.core_features:
            nli_res, _, _ = step_2_nli_matrix(p, feat)
            if nli_res != "CONTRADICTION":
                surviving_core.append(feat)

        # Simulate Gate 2 Pruning stats for Precision
        total_gaps_proposed = len(extraction.orthogonal_gaps) if extraction.orthogonal_gaps else 0
        valid_gaps_passed = 1 if total_gaps_proposed > 0 else 0 # Simplified for calculation

        gap_options = extraction.orthogonal_gaps[0].options[0:1] if extraction.orthogonal_gaps else []
        final_space = surviving_core + gap_options

        # Calculate Math for this run
        recall = min(len(surviving_core) / max(total_triplets, 1), 1.0)
        precision = 1.0 if total_gaps_proposed == 0 else (valid_gaps_passed / total_gaps_proposed)

        return final_space, recall, precision, triplets

    # 1. RUN EXPERIMENTS
    print(" -> Processing Base Prompt (Run 1)...")
    space_base_1, recall, precision, base_triplets = process_prompt_for_eval(prompt_base)

    print(" -> Processing Base Prompt (Run 2) for MU...")
    space_base_2, _, _, _ = process_prompt_for_eval(prompt_base)

    print(" -> Processing Rephrased Prompt for AS...")
    space_rephrased, _, _, _ = process_prompt_for_eval(prompt_rephrased)

    print(" -> Processing Different Prompt for IS...")
    space_different, _, _, _ = process_prompt_for_eval(prompt_different)

    # 2. CALCULATE METRICS
    # Tier 1
    mu = calculate_semantic_variance(space_base_1, space_base_2)
    as_score = calculate_semantic_variance(space_base_1, space_rephrased)
    is_score = calculate_semantic_variance(space_base_1, space_different)
    ici = is_score / (as_score + mu + 0.01)

    # Base Architecture Metrics (F1)
    # Using the exact IntentGrasp formula: 2 * (P * R) / (P + R)
    f1_score = 0.0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

    # Tier 2 (Prover/Refuter)
    print(" -> Running Formal Verification (Prover & Refuter)...")
    formal_spec_string = "\n".join(space_base_1)
    prover_pass, prover_reason = run_prover(prompt_base, formal_spec_string)
    refuter_pass, refuter_reason = run_refuter(prompt_base, formal_spec_string, base_triplets)
    scr = 1.0 if (prover_pass and refuter_pass) else 0.0

    # 3. PRINT MASTER DASHBOARD
    print(f"\n\n{'='*95}")
    print(f"🎓 BTP FINAL EVALUATION DASHBOARD: NEURO-SYMBOLIC REQUIREMENTS PIPELINE")
    print(f"{'='*95}")

    print(f"\n[PART 1: ARCHITECTURAL HARMONY (The 'IntentGrasp' Metrics)]")
    print(f"-----------------------------------------------------------------------------------------------")
    print(f"• Deterministic Intent Recall (DIR) : {recall*100:.1f}%  (Step 0 SRL Anchoring Success)")
    print(f"• Spurious Constraint Rejection     : {precision*100:.1f}%  (Step 2/3 NLI & AMR Pruning Success)")
    print(f"► F1 Harmony Score                  : {f1_score:.3f}  (Perfect balance of Completeness & Safety)")

    print(f"\n[PART 2: SEMANTIC STABILITY (Tier 1 Variance Math)]")
    print(f"-----------------------------------------------------------------------------------------------")
    print(f"• Model Uncertainty (MU)            : {mu:.3f}   (Target: 0.0 - Pipeline Determinism)")
    print(f"• Articulation Sensitivity (AS)     : {as_score:.3f}   (Target: 0.0 - Syntactic Immunity)")
    print(f"• Intent Sensitivity (IS)           : {is_score:.3f}   (Target: 1.0 - Dynamic Responsiveness)")
    print(f"► Intent Comprehension Index (ICI)  : {ici:.2f}   (Global Score)")

    print(f"\n[PART 3: FORMAL VERIFICATION (Tier 2 Adversarial Cascade)]")
    print(f"-----------------------------------------------------------------------------------------------")
    print(f"• Specification Prover (Plausibility) : {'✅ PASS' if prover_pass else '❌ FAIL'}")
    print(f"  └─ Reason: {prover_reason[:75]}...")
    print(f"• Adversarial Refuter (Omissions)     : {'✅ PASS' if refuter_pass else '❌ FAIL'}")
    print(f"  └─ Reason: {refuter_reason[:75]}...")
    print(f"► Spec Certification Rate (SCR)       : {scr:.1f}")
    print(f"{'='*95}\n")

# --- EXECUTE ---
prompt_base = "The system must decrypt the payload and update the ledger."
prompt_rephrased = "Before proceeding, the incoming payload is decrypted by the security module, and the transaction ledger gets updated."
prompt_different = "The system must authenticate the user and lock the database."

run_master_thesis_benchmark(prompt_base, prompt_rephrased, prompt_different)

🚀 INITIATING MASTER BTP EVALUATION SUITE...
 -> Processing Base Prompt (Run 1)...
 -> Processing Base Prompt (Run 2) for MU...
 -> Processing Rephrased Prompt for AS...
 -> Processing Different Prompt for IS...
 -> Running Formal Verification (Prover & Refuter)...


🎓 BTP FINAL EVALUATION DASHBOARD: NEURO-SYMBOLIC REQUIREMENTS PIPELINE

[PART 1: ARCHITECTURAL HARMONY (The 'IntentGrasp' Metrics)]
-----------------------------------------------------------------------------------------------
• Deterministic Intent Recall (DIR) : 100.0%  (Step 0 SRL Anchoring Success)
• Spurious Constraint Rejection     : 50.0%  (Step 2/3 NLI & AMR Pruning Success)
► F1 Harmony Score                  : 0.667  (Perfect balance of Completeness & Safety)

[PART 2: SEMANTIC STABILITY (Tier 1 Variance Math)]
-----------------------------------------------------------------------------------------------
• Model Uncertainty (MU)            : 0.000   (Target: 0.0 - Pipeline Determinism)
• Articulation Sensitivity

In [ ]:
# =====================================================================
# THE FINAL DEFENSE TEST SUITE (Financial/Security Domain)
# =====================================================================

# 1. THE BASE PROMPT (Clean, Active Voice, Strict Engineering)
prompt_base = (
    "When a client submits a market order, the matching engine must validate the "
    "account balance, execute the trade against the central limit order book, "
    "and notify the risk management module."
)

# 2. THE REPHRASED PROMPT (Messy, Passive Voice, Agile-style)
# This tests your Articulation Sensitivity (AS). It means the exact same thing,
# but the grammar is an absolute nightmare for a standard LLM to parse.
prompt_rephrased = (
    "Once the market order has been submitted by the client, checking the account "
    "balance must be done by the matching engine, and after the trade gets executed "
    "against the central limit order book, a notification should immediately be sent "
    "to the risk management module by the system."
)

# 3. THE DIFFERENT PROMPT (Domain Shift)
# This tests your Intent Sensitivity (IS). It forces the system to abandon
# the trading logic and generate a completely different security state-space.
prompt_different = (
    "Upon detecting a brute-force attack, the security firewall must isolate the "
    "server subsystem, encrypt the local user database, and generate an incident report."
)

# --- EXECUTE ---
run_master_thesis_benchmark(prompt_base, prompt_rephrased, prompt_different)

🚀 INITIATING MASTER BTP EVALUATION SUITE...
 -> Processing Base Prompt (Run 1)...
 -> Processing Base Prompt (Run 2) for MU...
 -> Processing Rephrased Prompt for AS...
 -> Processing Different Prompt for IS...
 -> Running Formal Verification (Prover & Refuter)...


🎓 BTP FINAL EVALUATION DASHBOARD: NEURO-SYMBOLIC REQUIREMENTS PIPELINE

[PART 1: ARCHITECTURAL HARMONY (The 'IntentGrasp' Metrics)]
-----------------------------------------------------------------------------------------------
• Deterministic Intent Recall (DIR) : 100.0%  (Step 0 SRL Anchoring Success)
• Spurious Constraint Rejection     : 50.0%  (Step 2/3 NLI & AMR Pruning Success)
► F1 Harmony Score                  : 0.667  (Perfect balance of Completeness & Safety)

[PART 2: SEMANTIC STABILITY (Tier 1 Variance Math)]
-----------------------------------------------------------------------------------------------
• Model Uncertainty (MU)            : 0.700   (Target: 0.0 - Pipeline Determinism)
• Articulation Sensitivity